# 02 - Tokenization: turning text into numbers

A neural network only does arithmetic; it cannot read the letter "h" directly. So the first step of every language model is to translate text into numbers. Each small piece of text is given a **token id**, which is simply a whole number that stands in for it. A useful image is a cloakroom: every distinct item is given a numbered ticket, the same item always receives the same number, and the ticket can later be exchanged back for the item.

We will start with the simplest scheme, one id per character, and then look briefly at what production models do, a method called subword **BPE** that is explained later in this notebook.

In [1]:
# Colab setup -- fetch the files this notebook needs.
# (Does nothing when run locally in the course folder.)
import os, urllib.request
BASE = ("https://raw.githubusercontent.com/waze"
        "emlabs/llm-book-code/main/")
for f in ['data/input.txt']:
    if not os.path.exists(f):
        d = os.path.dirname(f)
        if d: os.makedirs(d, exist_ok=True)
        urllib.request.urlretrieve(BASE + f, f)
        print("downloaded", f)


<details class="code-read"><summary>Line by line: what each line does</summary>
<ul>
<li><code>import os, urllib.request</code>: two standard-library toolboxes: checking files on disk, and downloading from the web.</li>
<li><code>BASE = (...)</code>: the web address of this course's folder on GitHub, split over two lines (Python glues adjacent strings together).</li>
<li><code>for f in [...]</code>: loop over the file names this notebook needs.</li>
<li><code>if not os.path.exists(f)</code>: only download what is missing -- running locally, everything already exists, so nothing happens.</li>
<li><code>os.makedirs(d, exist_ok=True)</code>: create the folder for the file if it has one (<code>exist_ok</code> means don't complain if it's already there).</li>
<li><code>urllib.request.urlretrieve(...)</code>: download the file and save it under the same name here.</li>
</ul>
</details>

In [2]:
from pathlib import Path
text = Path("data/input.txt").read_text()
print("dataset characters:", len(text))
print("----- first 200 chars -----")
print(text[:200])

dataset characters: 1115394
----- first 200 chars -----
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


<details class="code-read"><summary>Line by line: what each line does</summary>
<ul>
<li><code>from pathlib import Path</code>: borrow just the <code>Path</code> tool (for working with files) from Python's standard library.</li>
<li><code>text = Path("data/input.txt").read_text()</code>: open that file and hand back its entire contents as one long string. <code>text</code> now holds all ~1.1 million characters of Shakespeare.</li>
<li><code>len(text)</code>: count the characters in the string.</li>
<li><code>text[:200]</code>: slice the first 200 characters (positions 0 up to, but not including, 200) for a quick peek.</li>
</ul>
</details>

## A character-level tokenizer

The **vocabulary** is just the list of distinct characters that appear in the text (here there are 65 of them: the letters, the space, punctuation, and the newline). We number them and build two small lookup tables:

- `stoi`, short for "string to int": give it a character and it returns that character's id.
- `itos`, short for "int to string": the reverse, turning an id back into its character.

`encode` runs a whole string through `stoi` to produce a list of ids, and `decode` runs ids back through `itos` to rebuild the string. The `assert` line is a self-check that the round trip is lossless, meaning that encoding and then decoding returns exactly what you started with. For example, `encode("hi there")` becomes `[46, 47, 1, 58, 46, 43, 56, 43]`, where the `1` in the middle is the id for the space.

In [3]:
chars = sorted(set(text))
vocab_size = len(chars)
print("vocab size:", vocab_size)
print("vocab:", "".join(chars))

stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)

print("encode('hi there') ->", encode("hi there"))
print("round-trip:", decode(encode("hi there")))
assert decode(encode("First Citizen")) == "First Citizen"
print("round-trip OK")

vocab size: 65
vocab: 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
encode('hi there') -> [46, 47, 1, 58, 46, 43, 56, 43]
round-trip: hi there
round-trip OK


<details class="code-read"><summary>Line by line: what each line does</summary>
<ul>
<li><code>chars = sorted(set(text))</code>: <code>set(text)</code> throws away duplicates, leaving the distinct characters; <code>sorted</code> puts them in a fixed order. That ordered list of 65 characters is the vocabulary.</li>
<li><code>vocab_size = len(chars)</code>: how many distinct characters there are: 65.</li>
<li><code>stoi = {c: i for i, c in enumerate(chars)}</code>: build a lookup table ("string to int") in one line: <code>enumerate</code> walks the list handing out (position, character) pairs, and each character is filed under its position. So <code>stoi['a']</code> gives a's id.</li>
<li><code>itos = {i: c for c, i in stoi.items()}</code>: the reverse table ("int to string"), built by flipping every (char, id) pair.</li>
<li><code>encode = lambda s: [stoi[c] for c in s]</code>: a one-line function (<code>lambda</code>): walk a string, look each character up in <code>stoi</code>, collect the ids into a list.</li>
<li><code>decode = lambda ids: "".join(itos[i] for i in ids)</code>: the reverse: turn each id back into its character and glue them into one string (<code>"".join</code> concatenates with nothing between).</li>
<li><code>encode("hi there")</code>: shows the ids; the <code>1</code> in the result is the space.</li>
<li><code>assert decode(encode("First Citizen")) == "First Citizen"</code>: a tripwire: if encode-then-decode ever fails to return the original, the notebook stops loudly right here instead of going wrong silently later.</li>
</ul>
</details>

In [4]:
import numpy as np
data = np.array(encode(text), dtype=np.int64)
print("encoded dataset:", data.shape, data.dtype)
print("first 30 ids:", data[:30])

# Hold out the last 10% as a validation set (to check we're not just memorizing).
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]
print("train:", len(train_data), " val:", len(val_data))

encoded dataset: (1115394,) int64
first 30 ids: [18 47 56 57 58  1 15 47 58 47 64 43 52 10  0 14 43 44 53 56 43  1 61 43
  1 54 56 53 41 43]
train: 1003854  val: 111540


<details class="code-read"><summary>Line by line: what each line does</summary>
<ul>
<li><code>import numpy as np</code>: the array toolbox again.</li>
<li><code>data = np.array(encode(text), dtype=np.int64)</code>: encode the whole book into ids and store them as a NumPy array of 64-bit integers (<code>dtype</code> = what kind of number).</li>
<li><code>data.shape</code> / <code>data[:30]</code>: confirm it's one long sequence of ~1.1M ids, and peek at the first 30.</li>
<li><code>n = int(0.9 * len(data))</code>: the index 90% of the way through. <code>int(...)</code> chops it to a whole number.</li>
<li><code>train_data, val_data = data[:n], data[n:]</code>: slice into two parts: everything before <code>n</code> to train on, everything from <code>n</code> onward held back as a validation set (the practice exam).</li>
</ul>
</details>

## What real models do: subword (BPE) tokenization

Character-level tokenizing is simple but wasteful. The word "Tokenization" becomes 12 separate ids, and a single letter carries very little meaning on its own, so the model has to work harder over longer sequences.

Production language models instead use **Byte-Pair Encoding**, usually shortened to **BPE**. The idea is straightforward: scan the text, find the most common neighbouring pair of pieces, glue that pair into one new token, and repeat this thousands of times. Common chunks such as `" the"` or `"ing"` end up as a single id each. The vocabulary grows to roughly 50,000 to 100,000 tokens, but any given sentence becomes far shorter.

You can see the benefit below: the same sentence is 37 character-ids but only **7** GPT-2 tokens, and the pieces (`"Token"`, `"ization"`, `" splits"`, and so on) line up with how a person would naturally break the sentence up. We will keep using character-level tokens for the rest of the course because they are clearer to follow, but this is what the production systems do.

In [5]:
import tiktoken
enc = tiktoken.get_encoding("gpt2")     # GPT-2's actual BPE tokenizer
sample = "Tokenization splits text into pieces."
print("char-level ids :", len(encode(sample)), "tokens")
print("GPT-2 BPE ids  :", len(enc.encode(sample)), "tokens ->", enc.encode(sample))
print("BPE pieces     :", [enc.decode([t]) for t in enc.encode(sample)])

char-level ids : 37 tokens
GPT-2 BPE ids  : 7 tokens -> [30642, 1634, 30778, 2420, 656, 5207, 13]
BPE pieces     : ['Token', 'ization', ' splits', ' text', ' into', ' pieces', '.']


<details class="code-read"><summary>Line by line: what each line does</summary>
<ul>
<li><code>import tiktoken</code>: OpenAI's real tokenizer library (this needs installing, which is why this cell runs in the notebook, not the browser).</li>
<li><code>enc = tiktoken.get_encoding("gpt2")</code>: load GPT-2's actual trained BPE vocabulary.</li>
<li><code>len(encode(sample))</code>: our character tokenizer's count for the sentence (37).</li>
<li><code>enc.encode(sample)</code>: GPT-2's ids for the same sentence: only 7, because it merged common chunks into single tokens.</li>
<li><code>[enc.decode([t]) for t in enc.encode(sample)]</code>: decode each id <i>individually</i> to reveal the text chunk it stands for. Note <code>[t]</code>: <code>decode</code> wants a list, so each id is wrapped in one.</li>
</ul>
</details>

## Context windows and batches

We never feed the whole book in at once. It is too large, and the model only needs the recent context to predict the next character. So we cut the text into fixed-length chunks of `block_size` tokens. Such a chunk is called the **context window**: the stretch of text the model can see at one time.

There is a neat point about the labels. Within a single chunk, the target at every position is simply the next token. So one 8-character chunk quietly contains 8 training examples at once: given `t` predict `h`, given `th` predict a space, given `th ` predict `s`, and so on. The unrolled printout below shows exactly this.

That one relationship, context in and next token out, is the entire training signal. Nobody hand-labels anything; the text itself is the answer key. This is why language models can be trained on essentially all the text in the world.

A **batch** is just several of these chunks stacked together (`batch_size` of them), so the computer can process many examples at the same time. That is all `get_batch` does: it picks a few random starting points, slices out the context as `x`, and slices out the same span shifted one step to the right as the targets `y`.

In [6]:
def get_batch(split, block_size=8, batch_size=4, seed=None):
    data = train_data if split == "train" else val_data
    rng = np.random.default_rng(seed)
    ix = rng.integers(0, len(data) - block_size, size=batch_size)
    x = np.stack([data[i:i+block_size] for i in ix])        # (batch, block) context
    y = np.stack([data[i+1:i+1+block_size] for i in ix])    # (batch, block) the next token at each step
    return x, y

xb, yb = get_batch("train", block_size=8, batch_size=4, seed=1)
print("x batch shape:", xb.shape, " y batch shape:", yb.shape)
print("\nWhat one row means (context -> next token), unrolled:")
for t in range(8):
    ctx = xb[0, :t+1]
    print(f"  given {repr(decode(ctx)):20} predict {repr(decode([yb[0, t]]))}")

x batch shape: (4, 8)  y batch shape: (4, 8)

What one row means (context -> next token), unrolled:
  given 't'                  predict 'h'
  given 'th'                 predict ' '
  given 'th '                predict 's'
  given 'th s'               predict 't'
  given 'th st'              predict 'o'
  given 'th sto'             predict 'l'
  given 'th stol'            predict "'"
  given "th stol'"           predict 'n'


<details class="code-read"><summary>Line by line: what each line does</summary>
<ul>
<li><code>def get_batch(split, block_size=8, batch_size=4, seed=None):</code>: grab a training batch; the defaults mean you can just call <code>get_batch("train")</code>.</li>
<li><code>data = train_data if split == "train" else val_data</code>: a one-line if/else picking which dataset to draw from.</li>
<li><code>rng = np.random.default_rng(seed)</code>: a random-number generator; passing a seed makes the picks repeatable.</li>
<li><code>ix = rng.integers(0, len(data) - block_size, size=batch_size)</code>: choose 4 random starting positions, kept far enough from the end that a full 8-long chunk fits.</li>
<li><code>x = np.stack([data[i:i+block_size] for i in ix])</code>: slice out an 8-character chunk at each start and stack the 4 chunks into a 4&times;8 grid. This is the context.</li>
<li><code>y = np.stack([data[i+1:i+1+block_size] for i in ix])</code>: the same chunks shifted one step right: at every position, <code>y</code> holds the character that comes <i>next</i>. That's the answer key.</li>
<li><code>for t in range(8): ... print(...)</code>: unroll one chunk to show the eight (context &rarr; next-character) lessons hiding inside it.</li>
<li><code>repr(decode(ctx))</code>: show the decoded text with quotes and escape characters visible, so spaces and newlines are unmistakable.</li>
</ul>
</details>

## Look inside one batch

The shape line says `(4, 8)` and the eight lessons come from the first chunk. Both are worth seeing in full, because this grid is the shape every later notebook works in, and "a grid of ids" stays vague until you have looked at one. Start with the four random positions:

In [7]:
rng = np.random.default_rng(1)
ix = rng.integers(0, len(train_data) - 8, size=4)
print("ix:", ix)


ix: [475008 513790 758071 954119]


<details class="code-read"><summary>Line by line: what each line does</summary>
<ul>
<li><code>rng = np.random.default_rng(1)</code>: a seeded random-number generator, so these four positions come out the same every run.</li>
<li><code>ix = rng.integers(0, len(train_data) - 8, size=4)</code>: four random places to start reading, each kept far enough from the end that a full 8-character chunk still fits.</li>
<li><code>print(&quot;ix:&quot;, ix)</code>: nothing clever happened: these are just four offsets scattered through a million-character book.</li>
</ul>
</details>

Each of those becomes one row of the batch:

In [8]:
xb, yb = get_batch("train", block_size=8,
                   batch_size=4, seed=1)
for r in range(4):
    print(repr(decode(xb[r])), "->",
          repr(decode(yb[r])))


"th stol'" -> "h stol'n"
'ome, tho' -> 'me, thou'
'e jot be' -> ' jot bey'
's as see' -> ' as seem'


<details class="code-read"><summary>Line by line: what each line does</summary>
<ul>
<li><code>xb, yb = get_batch(...)</code>: draw one batch: four chunks of eight characters, and their answer key.</li>
<li><code>for r in range(4):</code>: walk the four rows one at a time.</li>
<li><code>print(repr(decode(xb[r])), &quot;-&gt;&quot;, repr(decode(yb[r])))</code>: turn each row of ids back into text so the pairing is readable. Every pair is a scrap of Shakespeare beside itself shifted one character along &mdash; given <code>th stol'</code>, the answer is <code>h stol'n</code>.</li>
</ul>
</details>

Underneath the text it is all numbers:

In [9]:
print(xb)


[[58 46  1 57 58 53 50  5]
 [53 51 43  6  1 58 46 53]
 [43  1 48 53 58  1 40 43]
 [57  1 39 57  1 57 43 43]]


<details class="code-read"><summary>Line by line: what each line does</summary>
<ul>
<li><code>print(xb)</code>: the <code>(4, 8)</code> the shape line reported: four rows, eight ids each. Every number is a cloakroom ticket from the tokenizer &mdash; 58 is <code>t</code>, 46 is <code>h</code>, 1 is the space &mdash; which is why row 0 reads <code>th stol'</code>.</li>
</ul>
</details>

And the "shifted by one" claim can be checked instead of believed:

In [10]:
print("xb[0][1:] :", xb[0][1:])
print("yb[0][:-1]:", yb[0][:-1])
print("same?      ",
      bool((xb[0][1:] == yb[0][:-1]).all()))


xb[0][1:] : [46  1 57 58 53 50  5]
yb[0][:-1]: [46  1 57 58 53 50  5]
same?       True


<details class="code-read"><summary>Line by line: what each line does</summary>
<ul>
<li><code>xb[0][1:]</code>: row 0 of the context with its <i>first</i> id dropped.</li>
<li><code>yb[0][:-1]</code>: row 0 of the answer key with its <i>last</i> id dropped.</li>
<li><code>bool((xb[0][1:] == yb[0][:-1]).all())</code>: <code>True</code>: what is left is identical. Read the check slowly, because <code>==</code> behaves differently here than you may expect: comparing two NumPy arrays does not give one answer, it compares slot by slot and hands back an array of <code>True</code>/<code>False</code>, one per position. <code>.all()</code> collapses that array to a single verdict &mdash; "were they <i>all</i> True?" That overlap is the whole reason a chunk of 8 characters carries 8 lessons rather than 1 &mdash; <code>y</code> is not a separate answer key somebody wrote out, it is <code>x</code> itself, read one step later.</li>
</ul>
</details>

## Recap

The text is now a stream of whole-number ids, split into a training portion and a held-back validation portion, so that later we can check the model is genuinely learning the language rather than memorizing the book. We can draw `(context, next-token)` batches on demand. Next, notebook 03 uses these to train your first real language model.